# Unix-terminal. Работа с сетью


## Мотивация

Вы развернули что-то на сервере: запустили обучение с TensorBoard, подняли
Jupyter, отдали коллеге ссылку на свой сервис. Открываете в браузере —
«страница недоступна».

Дальше обычно начинается угадывание: перезапустить, поменять порт, написать
администратору, попробовать с телефона. Уходит час, причина так и не найдена,
а на следующей неделе всё повторяется.

Сеть устроена слоями, и **каждый слой проверяется отдельной командой за
секунды**. Если идти снизу вверх, ответ находится с третьей-четвёртой попытки,
а не с тридцатой — причём находится точное место поломки, а не «что-то с сетью».

Лестница диагностики

Ступени соотносятся с уровнями модели TCP/IP из лекции 6, но не повторяют их:
это порядок **проверки**, а не порядок стека. Хост и маршрут — сетевой уровень
(L3), порт — транспортный (L4), ответ сервиса — прикладной (L7). Имя стоит внизу
лестницы не потому, что оно ниже по модели (DNS — как раз прикладной протокол),
а потому, что пока имя не превратилось в адрес, проверять нечего.

Сегодня мы пройдём эту лестницу целиком. К концу занятия на вопрос «почему не
работает» у вас будет ответ вида «сломано на слое 4: порт слушается только на
loopback, вот вывод `ss`» — а с таким ответом чинится уже быстро.


### Рабочий каталог

Всё делаем на своей виртуальной машине в `~/seminar-07/`. Каталог `/tmp` не
берём: он вычищается при перезагрузке, а файлы нужны и после занятия.

Часть ячеек ходит в интернет: `example.com` — для DNS, кодов ответа и
сертификата, `1.1.1.1` — публичный резолвер, `ifconfig.me` — внешний адрес.
Разделы про HTTP и HTTPS работают на своих серверах и сети не требуют.


In [ ]:
%%bash
mkdir -p ~/seminar-07
cd ~/seminar-07 || exit 1
pwd


Семинару нужны четыре программы сверх базовой установки. Проверим сразу, а не
на середине занятия.


In [ ]:
%%bash
missing=""
for t in curl dig nc tcpdump openssl ufw; do
    command -v "$t" > /dev/null || missing="$missing $t"
done
[ -z "$missing" ] && echo "все инструменты на месте" && exit 0
echo "не хватает:$missing"
echo "поставить: sudo apt install curl dnsutils netcat-openbsd tcpdump openssl ufw"


## 1. Свой узел: адрес, интерфейс, маршрут

Нижняя ступень лестницы. Прежде чем спрашивать «почему не отвечает тот сервер»,
надо убедиться, что у нас самих есть адрес и понятно, куда пойдут пакеты.


In [ ]:
%%bash
ip -brief address


Три вещи в выводе: имя интерфейса, состояние и адреса. Состояний не два, а
три: `UP` у работающего внешнего интерфейса, `DOWN` у выключенного и `UNKNOWN`
у `lo` и туннелей — ядру нечего сообщать о «линке» там, где нет провода.

Кстати, привычное `ip a` — сокращение того же `ip address`; ключ `-brief`
только меняет формат вывода на табличный.
Интерфейс `lo` с адресом `127.0.0.1` — loopback: он есть всегда и никуда
наружу не ведёт. Внешний интерфейс обычно называется `eth0`, `ens3` или похоже.

Адресов у машины несколько, и это нормально. Какой из них «настоящий» —
зависит от того, куда мы собрались идти. Это и показывает `ip route get`.


In [ ]:
%%bash
ip route show default


Шлюз по умолчанию — узел, которому машина отдаёт всё, для чего нет более
точного маршрута. Это первая внешняя точка на пути: если её нет, наружу не
уйдёт ничего.

Строка читается по ключевым словам: после `via` стоит **адрес шлюза** (именно
он нужен в задаче B1), после `dev` — интерфейс, которым до шлюза идти, после
`src` — адрес, который машина подставит отправителем.

А куда пойдёт пакет к конкретному адресу, показывает `ip route get`. Сравним
дальний адрес и свой собственный.


In [ ]:
%%bash
ip route get 8.8.8.8
ip route get 127.0.0.1


Читается так: чтобы дойти до `8.8.8.8`, пакет пойдёт через шлюз `via ...`,
интерфейсом `dev ...`, и уйдёт **с адреса** `src ...`. Это адрес, с которым
пакет покидает нашу машину, — но не обязательно тот, который увидит собеседник:
по дороге его может подменить NAT, об этом следующий раздел.

У второй строчки `via` нет вовсе: до собственного адреса шлюз не нужен, пакет
никуда не уходит и остаётся на интерфейсе `lo`. Так же выглядит маршрут до
соседа по локальной сети — прямой, без шлюза.

#### ❓ **Вопрос**: У машины есть и `127.0.0.1`, и адрес вида `10.0.0.5`. По какому из них коллега с соседней машины сможет к вам обратиться?

<details>

<summary><strong>Ответ</strong></summary>

Только по `10.0.0.5` — тому, что показан в `src` у `ip route get`. Адрес
`127.0.0.1` есть у каждой машины свой собственный, он не выходит за пределы
хоста: коллега, набрав `127.0.0.1`, попадёт на самого себя — и это не ошибка
адреса, а именно то, что такой адрес означает.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

В старых инструкциях из интернета вы встретите `ifconfig`, `route` и `netstat`
— это пакет `net-tools`, он не развивается с середины 2000-х и в свежих
образах часто просто не установлен. Современная замена — `ip` (адреса,
маршруты, интерфейсы) и `ss` (сокеты) из пакета `iproute2`. Команды `ifconfig`
и `netstat` продолжают работать там, где их доставили, но новые ключи и типы
адресов они уже не знают, поэтому в курсе используем `ip` и `ss`.

</details>


## 2. Внешний адрес: почему их два

Адрес на интерфейсе и адрес, под которым машину видит интернет, — часто разные.
Между ними стоит NAT: домашний роутер или облачный шлюз.

NAT появился из-за нехватки адресов IPv4: их около четырёх миллиардов, и на всех
давно не хватает. Поэтому целой сети выдают один публичный адрес, а внутри
раздают частные, которые в интернете не маршрутизируются. В IPv6 адресов хватает
с запасом, и там NAT не нужен.

Про ключи ниже: `show scope global` отбрасывает служебные адреса (`lo` и
локальные для канала), оставляя те, по которым к машине в принципе можно
обратиться извне; `curl -4` просит идти по IPv4, иначе сервис-зеркало вернёт
адрес IPv6 и сравнивать будет неудобно.


In [ ]:
%%bash
ip -brief address show scope global
echo "--- а так меня видит интернет ---"
curl -4 -s -m 5 https://ifconfig.me
echo


#### ❓ **Вопрос**: `ip address` показывает `10.x.x.x`, а сервис в интернете отвечает совсем другим адресом. Кто из них врёт?

<details>

<summary><strong>Ответ</strong></summary>

Никто. `10.x.x.x` — частный адрес внутри локальной сети, он есть у миллионов
машин одновременно и в интернете не маршрутизируется. На выходе роутер
подменяет адрес отправителя на свой публичный — это и есть NAT. Практическое
следствие: снаружи к вашему `10.x.x.x` подключиться нельзя, даже если сервис
слушает на всех интерфейсах, — нужен проброс порта на роутере или туннель.

</details>


## 3. Имя: во что оно превращается

Вторая ступень. Пока имя не превратилось в адрес, соединяться не с чем.

Путь от имени к адресу


In [ ]:
%%bash
getent hosts example.com
dig +short example.com


`getent hosts` спрашивает так же, как это делают обычные программы — через
системную функцию `getaddrinfo`. `dig` — инструмент отладки DNS: он ходит в
DNS-сервер напрямую и показывает сырой ответ.

Если у машины есть IPv6, `getent hosts` вполне может вернуть адрес вида
`2606:4700::...`, а `dig +short` — привычный IPv4. Это не противоречие:
`getaddrinfo` отдаёт программе **список** адресов обоих семейств, а `dig` без
уточнений спрашивает только записи типа A, то есть IPv4.

Полный вывод `dig` полезен двумя строчками: сама A-запись с временем жизни
(TTL) и `SERVER:` — кто именно ответил. Ключи `+noall +answer` оставляют от
ответа только секцию с записями, сколько бы их ни было.

Если `dig` не установлен (в минимальных образах его нет), поставьте пакет:
`sudo apt install dnsutils`.


In [ ]:
%%bash
dig example.com +noall +answer
dig example.com | grep 'SERVER:'


Кого именно спрашивает система, написано в `/etc/resolv.conf`.


In [ ]:
%%bash
grep -v -e '^#' -e '^$' /etc/resolv.conf
echo "--- порядок источников имён ---"
grep '^hosts:' /etc/nsswitch.conf


Адрес `127.0.0.53` — это не «настоящий» DNS-сервер в интернете, а локальный
stub-резолвер `systemd-resolved` на нашей же машине. Он принимает запросы
программ, ходит к реальным серверам провайдера или облака, кэширует ответы и
знает про локальные имена. Отсюда практическое следствие: строка `SERVER:` в
выводе `dig` почти всегда покажет именно его, а не тот сервер, который на самом
деле дал ответ.

Вторая строка вывода — из `/etc/nsswitch.conf`: `hosts: files dns` читается как
«сначала файл `/etc/hosts`, потом DNS». Этот порядок и решает, кто победит при
расхождении.


А теперь фокус, который объясняет половину загадочных случаев «у меня
резолвится, а программа не идёт»:


In [ ]:
%%bash
grep -w localhost /etc/hosts | head -2
echo "--- getent: так адрес получают программы ---"
getent hosts localhost
echo "--- dig у публичного DNS-сервера ---"
dig +short localhost @1.1.1.1
echo "(пусто — в публичном DNS имени localhost нет)"


#### ❓ **Вопрос**: `dig` показывает у сайта один адрес, а программа упорно идёт на другой. Что проверить первым делом и почему?

<details>

<summary><strong>Ответ</strong></summary>

`/etc/hosts` — и сверять надо `getent hosts`, а не `dig`. Программы ходят за
адресом через `getaddrinfo`, а тот сначала читает этот файл и, найдя запись,
в DNS уже не идёт. `dig` же спрашивает DNS всегда, поэтому и показывает не то,
что реально получит программа. Ровно это мы и видели на `localhost`: адрес есть,
хотя в публичном DNS имени нет.

Файл существует ровно для таких локальных переопределений: временно направить
имя на тестовый сервер, дать короткое имя машине в локальной сети, заблокировать
домен, отправив его на `127.0.0.1`. В стандартной настройке (строка `hosts: files dns` в
`/etc/nsswitch.conf`) он сильнее DNS — и именно поэтому оказывается причиной
«у всех работает, а у меня нет». Порядок источников задаётся именно там, и на
нетипичной машине его могли поменять.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Тонкость, на которой легко обжечься: `dig +short localhost` **без** указания
сервера на современной Ubuntu отвечает `127.0.0.1`. Кажется, что это опровергает
всё сказанное, но нет: запрос уходит в локальный stub-резолвер
`systemd-resolved` (адрес `127.0.0.53`), а тот синтезирует ответ для `localhost`
сам, не спрашивая интернет. Именно поэтому в демонстрации мы явно указали
`@1.1.1.1` — публичный сервер.

Практический вывод шире примера: у `dig` всегда стоит спрашивать себя, **какой
резолвер** ответил. Строка `SERVER:` в полном выводе — не украшение: ответы
локального кэша, корпоративного DNS и публичного сервера могут различаться, и
половина «мистических» расхождений живёт именно здесь.

</details>


## 4. Хост: доходят ли пакеты

Третья ступень. Адрес есть — проверяем, отвечает ли машина.


In [ ]:
%%bash
ping -c 3 -W 2 example.com


Ключи: `-c 3` — отправить три пакета и остановиться (без него `ping` будет
идти, пока его не прервут), `-W 2` — ждать ответ на каждый не дольше двух
секунд.

Смотрим на две вещи: процент потерь и время отклика. Потери — признак плохого
канала, большое время — признак дальнего или перегруженного маршрута.

Теперь постучимся по адресу из диапазона, который зарезервирован под примеры в
документации и в интернете не маршрутизируется:


In [ ]:
%%bash
ping -c 2 -W 2 192.0.2.1 || echo "ответа нет"


#### ❓ **Вопрос**: Сервер не отвечает на `ping`. Можно ли из этого сделать вывод, что он выключен?

<details>

<summary><strong>Ответ</strong></summary>

Нельзя. `ping` использует протокол ICMP, а его очень часто режут на фаерволах
и у облачных провайдеров — специально, чтобы машину не сканировали. Живой
сервер с работающим сайтом может молчать на `ping`.

Обратное утверждение сильнее: если `ping` **отвечает** — хост точно жив и
маршрут до него есть. Поэтому успешный `ping` — хорошая новость, а неуспешный
ничего не доказывает, и надо идти на ступень выше и стучаться прямо в порт.

</details>


## 5. Порт: кто слушает у меня

Четвёртая ступень. Адрес приводит пакет на нужную машину, но на машине работают
десятки программ — кому его отдать? Для этого к адресу добавлен **порт**:
16-битный номер (1–65535), по которому ядро понимает, какому приложению отдать
пришедшие данные. Программа, заявившая ядру «данные для порта 8000 — мои»,
называется слушающей, а сама заявка — **слушающим сокетом**.

Поднимем свой сервис, чтобы было что диагностировать.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
pkill -f "http.server 8000"
echo "привет из семинара 7" > index.html
nohup python3 -m http.server 8000 --bind 0.0.0.0 > server.log 2>&1 &
sleep 1
ss -tlnp | grep 8000


Две команды в ячейке заслуживают расшифровки. `pkill -f "http.server 8000"`
завершает процесс, найдя его **по полной командной строке** (`-f`), — так ячейку
можно перезапускать, не оставляя занятым порт с прошлого раза. `nohup ... &`
запускает сервер в фоне и отвязывает его от терминала, чтобы он пережил конец
ячейки; вывод при этом уходит в файл `server.log`.

Ключи `ss` разбираются по буквам: `-t` — TCP, `-l` — только слушающие сокеты,
`-n` — не превращать номера портов в имена, `-p` — показать процесс-владелец.

Главная колонка — `Local Address:Port`. Слева от двоеточия стоит **адрес
привязки**, и именно он решает, кто сможет подключиться.

#### ❓ **Вопрос**: В колонке адреса написано `0.0.0.0:8000`. Что означает `0.0.0.0` — это какой-то конкретный узел?

<details>

<summary><strong>Ответ</strong></summary>

Нет, это не адрес узла, а способ сказать «любой адрес этой машины». Сервис
принимает соединения на всех интерфейсах сразу: и на loopback `127.0.0.1`, и на
внешнем `10.x.x.x`. Противоположность — привязка к одному конкретному адресу,
и вот тогда начинаются сюрпризы (следующий раздел).

</details>


## 6. Ловушка: адрес привязки

Самая частая причина «на сервере работает, а из браузера нет». Порт открыт,
процесс жив, в логах пусто — а снаружи отказ.

Адрес привязки: 127.0.0.1 против 0.0.0.0

Вариантов привязки три: `127.0.0.1` — только изнутри машины, `0.0.0.0` — на
всех интерфейсах сразу и один конкретный адрес машины (`--bind 10.0.2.15`) —
когда интерфейсов несколько и сервис должен быть виден только в одной сети.
Перезапустим тот же сервер, поменяв только адрес привязки:


In [ ]:
%%bash
pkill -f "http.server 8000"
sleep 1
cd ~/seminar-07 || exit 1
nohup python3 -m http.server 8000 --bind 127.0.0.1 > server.log 2>&1 &
sleep 1
ss -tlnp | grep 8000


Порт по-прежнему слушается, процесс тот же. Проверим его с двух сторон: через
loopback и через собственный внешний адрес машины.

Адрес возьмём не глазами, а из маршрута: `grep -oP 'src \K\S+'` печатает то,
что стоит сразу после слова `src`. Здесь `-o` выводит только совпавшую часть,
`-P` включает перловые регулярные выражения, а `\K` выбрасывает из совпадения
всё, что было до него.


In [ ]:
%%bash
MY_IP=$(ip route get 8.8.8.8 | grep -oP 'src \K\S+')
echo "внешний адрес машины: $MY_IP"
curl -s -m 3 http://127.0.0.1:8000/ || echo "через loopback: не ответил"
curl -s -m 3 "http://$MY_IP:8000/" || echo "через внешний адрес: отказ"


#### ❓ **Вопрос**: Коллега говорит «твой сервис не открывается», вы заходите на сервер, делаете `curl localhost:8000` — всё работает. Что проверить первым делом?

<details>

<summary><strong>Ответ</strong></summary>

Адрес привязки в выводе `ss -tlnp`: стоит ли там `127.0.0.1:8000` вместо
`0.0.0.0:8000`. Проверка `curl localhost` в этой ситуации бесполезна — она
проходит через loopback и будет успешной в обоих случаях. Проверять надо по тому
же адресу, по которому идёт коллега.

Многие сервисы (Jupyter, TensorBoard, dev-серверы фреймворков) привязываются к
loopback **по умолчанию** — это защита, а не ошибка: так сервис не оказывается
случайно открыт всей сети.

Вариантов привязки не два, а три: кроме `127.0.0.1` и `0.0.0.0` можно указать
один конкретный адрес машины (`--bind 10.0.2.15`). Так делают, когда у сервера
несколько интерфейсов — например, внутренний и публичный, — и сервис должен
быть виден только в одной из сетей.

</details>


## 7. Три исхода стука в порт

Если запомнить из семинара одну вещь — пусть это будет она.

Три исхода TCP-подключения


In [ ]:
%%bash
nc -z -v -w 3 127.0.0.1 8000 2>&1 | tail -1
nc -z -v -w 3 127.0.0.1 9 2>&1 | tail -1
nc -z -v -w 3 192.0.2.1 80 2>&1 | tail -1


Ключи: `-z` — только проверить, ничего не передавать, `-v` — рассказывать, что
происходит, `-w 3` — не ждать дольше трёх секунд.

Две цели тут эталонные и пригодятся дальше. Порт 9 (`discard`) исторически
зарезервирован и на обычной машине никем не занят — гарантированный отказ. А
`192.0.2.1` — из диапазона, отведённого под примеры в документации (RFC 5737):
в интернете он не маршрутизируется, поэтому обычно даёт таймаут. Обычно, а не
всегда: если по дороге найдётся маршрутизатор, который вежливо ответит ICMP
unreachable, вместо ожидания получится мгновенный отказ.

Обратите внимание не только на текст, но и **на время**: первые две строки
появились мгновенно, третья — через три секунды.

#### ❓ **Вопрос**: В одном случае вы получили `Connection refused` сразу, в другом — «завис и отвалился по таймауту». Где чинить в каждом случае?

<details>

<summary><strong>Ответ</strong></summary>

`Connection refused` — до хоста мы дошли, кто-то ответил пакетом RST: «здесь
никто не слушает». Сеть и маршрут в порядке, чинить надо **на самом хосте**:
сервис не запущен, упал или слушает другой порт/адрес. Оговорка, к которой мы
вернёмся в разделе 9: отказ может прислать и фаервол с правилом REJECT — тогда
сервис на месте, а отвечает за него не он.

Таймаут — ответа не было вообще. Пакет где-то отбросили молча: фаервол с
правилом DROP, группа безопасности в облаке, неверный адрес, выключенная
машина. Чинить надо **по дороге**, а не в сервисе.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`nc` существует в нескольких несовместимых реализациях (openbsd, traditional,
ncat из состава nmap), и ключи у них расходятся: где-то нет `-z`, где-то
по-другому пишется таймаут. Проверять надо `nc -h`. В задаче H6 понадобится
ключ `-q 1` — «после конца ввода подождать секунду и выйти»; проверьте по
`nc -h`, что ваша сборка его понимает. Если `nc` вообще нет,
ту же проверку делает `curl -v telnet://host:port` или на голом bash:
`timeout 3 bash -c '</dev/tcp/127.0.0.1/8000'` — код возврата скажет то же
самое.

</details>


## 8. Таймаут: чей он и на каком шаге

В прошлом разделе таймаут был один — «ответа нет». На практике их различают
подробнее, и различие рабочее: по нему видно, **на каком шаге** оборвался
разговор и **кто** решил, что ждать хватит.

Три случая, которые надо уметь называть по имени:

- **таймаут подключения** — не удалось даже установить TCP-соединение;
- **таймаут ответа** — соединение установлено, запрос ушёл, ответа нет;
- **сервер сам разорвал соединение** — ждать надоело не нам, а ему.

У `curl` первый ограничивается ключом `--connect-timeout`, а общее время
запроса — ключом `--max-time`. Начнём с недостижимого адреса.


In [ ]:
%%bash
curl -s -o /dev/null --connect-timeout 3 --max-time 30 \
  -w 'соединение: %{time_connect} с, всего: %{time_total} с\n' \
  http://192.0.2.1/
echo "код возврата curl: $?"


`time_connect` равен нулю: соединения не случилось вовсе, все три секунды
ушли на ожидание ответа на первый же пакет. Код возврата `28` у `curl`
означает «истекло отведённое время».

Теперь поднимем сервер, который соединение **принимает**, но не отвечает
ничего. Роль такого «зависшего приложения» сыграет `nc`.


In [ ]:
%%bash
nc -l 127.0.0.1 8001 > /dev/null 2>&1 &
srv=$!
sleep 1
curl -s -o /dev/null --connect-timeout 3 --max-time 5 \
  -w 'соединение: %{time_connect} с, всего: %{time_total} с\n' \
  http://127.0.0.1:8001/
echo "код возврата curl: $?"
kill "$srv" 2>/dev/null || true


Код тот же `28`, а картина совсем другая: `time_connect` — тысячные доли
секунды, всё остальное время ушло на ожидание ответа.

#### ❓ **Вопрос**: Соединение установилось мгновенно, а ответа нет пять секунд. На какой ступени лестницы искать причину?

<details>

<summary><strong>Ответ</strong></summary>

На пятой, в самом приложении. Мгновенный `time_connect` доказывает, что имя,
маршрут, порт и фаервол в порядке: TCP-соединение установлено. Молчит уже сам
сервис — ждёт базу, ушёл в бесконечный цикл, не хватает воркеров. Смотреть надо
его логи и нагрузку, а не сеть.

Отсюда практическое правило: у таймаута всегда спрашивайте, **на каком шаге** он
случился. `--connect-timeout` и `--max-time` (в библиотеках — connect timeout и
read timeout) задаются отдельно именно потому, что означают разные поломки.

</details>


Третий случай: ждать надоело не клиенту, а серверу. Пусть наш `nc` закроется
сам через две секунды, а `curl` при этом готов ждать хоть тридцать. Ограничивает
время команды `timeout N` — она запускает программу и убивает её через N секунд.


In [ ]:
%%bash
timeout 2 nc -l 127.0.0.1 8001 > /dev/null 2>&1 &
sleep 1
curl -s -o /dev/null --connect-timeout 3 --max-time 30 \
  -w 'всего: %{time_total} с\n' http://127.0.0.1:8001/
echo "код возврата curl: $?"


Код `52` — «пустой ответ от сервера»: соединение закрылось раньше, чем пришла
хоть одна строка. Ждали меньше секунды, хотя клиент был готов ждать тридцать, — значит,
решение прекратить ожидание принял не он.

Итого три исхода и три разных диагноза:

| Что показал `curl` | Кто прекратил ожидание | Где чинить |
|---|---|---|
| код 28, `time_connect` = 0 | клиент, не дождавшись соединения | сеть: адрес, маршрут, фаервол |
| код 28, `time_connect` мал | клиент, не дождавшись ответа | приложение: логи, нагрузка |
| код 52 или обрыв | сервер | настройки таймаутов на сервере |


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Серверный таймаут почти всегда настраивается явно, и в вежливом варианте
клиент получает не обрыв, а честный код: `504 Gateway Timeout` от обратного
прокси означает «я спросил приложение, оно не ответило вовремя». Типичные
ручки: `proxy_connect_timeout` и `proxy_read_timeout` у nginx, `--timeout` у
gunicorn. Поэтому `504` — это **не** ваша сеть и не ваш клиент: это внутренний
таймаут на той стороне. Живьём мы его не показываем: нужен обратный прокси
перед приложением, а у нас приложение отвечает напрямую — поэтому в демке
получился код 52, «оборвал соединение», а не вежливый `504`.

Обратная сторона — клиентский таймаут. По умолчанию его во многих библиотеках
нет вовсе: `requests.get(url)` без `timeout=` может висеть, пока соединение не
разорвёт операционная система, и утащить за собой весь скрипт. Правило простое:
у каждого сетевого вызова в коде должен стоять таймаут, и лучше двумя числами —
`requests.get(url, timeout=(3, 10))`: три секунды на соединение, десять на
ответ.

</details>


## 9. Кто мешает: фаервол

Мы видели таймаут на неотвечающем адресе. Точно такой же таймаут получится,
если пакет отбросит фаервол на пути. Посмотреть правила на своей машине можно
так — команда требует `sudo`, поэтому её показывает преподаватель:


In [ ]:
%%bash
command -v ufw > /dev/null || { echo "ufw не установлен: sudo apt install ufw"; exit 0; }
sudo -n true 2>/dev/null || { echo "нужен sudo — команду показывает преподаватель"; exit 0; }
sudo ufw status verbose


Важная деталь про то, как фаервол отказывает. Есть два разных действия:

- **DROP** — пакет молча выбрасывается, отправителю не отвечают ничего.
  Клиент видит **таймаут**;
- **REJECT** — в ответ отправляется явный отказ. Клиент видит
  **connection refused**, как если бы сервис просто не был запущен.

По умолчанию `ufw deny` работает как DROP.

#### ❓ **Вопрос**: Почему администраторы обычно выбирают DROP, а не REJECT, хотя REJECT честнее и удобнее для отладки?

<details>

<summary><strong>Ответ</strong></summary>

Молчание замедляет сканирование: чтобы перебрать порты, атакующему придётся
ждать таймаут на каждом, вместо мгновенного ответа. Плюс на отказы тратится
исходящий трафик.

Обратная сторона — ровно та боль, ради которой мы сегодня собрались: молчание
неотличимо от молчания. «Фаервол с DROP», «машины нет», «неверный адрес» и
«пакет потерялся по дороге» снаружи выглядят одинаково — таймаутом, и по нему
нельзя понять, на какой ступени сломано. REJECT в этом смысле честнее, но и он
подменяет диагноз: его `refused` не отличить от незапущенного сервиса.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Синтаксис, который понадобится в задаче H3, целиком укладывается в четыре
команды:

```
sudo ufw deny 8000/tcp          # молча отбрасывать (DROP)
sudo ufw reject 8000/tcp        # отвечать явным отказом (REJECT)
sudo ufw delete deny 8000/tcp   # снять правило
sudo ufw status numbered        # посмотреть, что вообще настроено
```

Первая же грабля: на свежей Ubuntu `ufw status` показывает `inactive`, и в
таком состоянии правила добавляются, но ничего не фильтруют. Перед
экспериментом фаервол надо включить — и, если на машину ходят по SSH, сначала
разрешить SSH, иначе включение отрежет вас от собственного сервера:

```
sudo ufw allow OpenSSH     # ОБЯЗАТЕЛЬНО до включения, если работаете по ssh
sudo ufw enable
...                        # эксперимент
sudo ufw disable           # вернуть исходное состояние
```

Ожидаемая разница со стороны клиента: с `deny` вызов
`nc -z -v -w 5 АДРЕС 8000` висит все пять секунд и заканчивается `timed out`;
с `reject` — отвечает мгновенно `Connection refused`, то есть выглядит ровно
как незапущенный сервис.

Важная оговорка, из-за которой эксперимент часто «не работает»: стучаться надо
**с другой машины**. Трафик к собственному адресу идёт через `lo`, а `ufw`
пропускает `lo` правилом `-i lo -j ACCEPT` раньше всех пользовательских правил,
поэтому со своей же машины запрет не сработает и порт останется открытым.

</details>


## 10. Приложение: разговор с сервисом

Верхняя ступень. Соединение устанавливается — теперь смотрим, что отвечает сам
сервис. Здесь работает `curl`. Начнём со своего сервера, который всё это время
слушает `127.0.0.1:8000`.


In [ ]:
%%bash
curl -sS -I -m 5 http://127.0.0.1:8000/ | head -3


`-I` запрашивает только заголовки, без тела. Первая строка — код ответа.

Теперь то же самое, но к сайту в интернете: там добавятся TLS и настоящая
задержка, и на них удобно смотреть дальше.


In [ ]:
%%bash
curl -sS -I -m 5 https://example.com | head -4


Тот же ключ, но у сайта в интернете. Заголовки те же по смыслу, а значения
другие: версия протокола `HTTP/2` вместо `HTTP/1.0`, в `server:` вместо нашего
`SimpleHTTP/0.6 Python/3.12.3` — чужой веб-сервер. Код ответа в первой строке
сразу говорит, чья это проблема.


In [ ]:
%%bash
curl -s -o /dev/null -m 5 -w 'существующая страница: %{http_code}\n' https://example.com/
curl -s -o /dev/null -m 5 -w 'выдуманный адрес:      %{http_code}\n' https://example.com/net-2026


Те два запроса ушли наружу, а вот свой сервер пишет всё, что у него просили, в
`server.log` — тот файл, в который мы направили его вывод при запуске. Сходим к
нему на существующий и на выдуманный путь и посмотрим на лог с той стороны: на
пятой ступени лестницы это первое, куда надо заглядывать.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
curl -s -o /dev/null http://127.0.0.1:8000/
curl -s -o /dev/null http://127.0.0.1:8000/net-2026
sleep 1
tail -3 server.log


В логе видно время, метод, путь и код: `200` на существующий путь и `404` на
выдуманный — то же самое, что видел `curl`, но с другой стороны. Отсюда главный
приём пятой ступени: если клиент жалуется, а в логе сервиса пусто, значит запрос
до приложения не дошёл — возвращаемся на ступень ниже искать, где его потеряли.


#### ❓ **Вопрос**: Сервис вернул `404`. На какой ступени лестницы проблема?

<details>

<summary><strong>Ответ</strong></summary>

Ни на какой из нижних — все они пройдены успешно. Имя разрешилось, хост
достижим, порт слушается, сервис принял запрос, понял его и осмысленно
ответил «такого пути у меня нет». Это проблема на уровне приложения: неверный
URL или не задеплоенный обработчик.

Ровно поэтому `404` — хорошая новость при диагностике: он доказывает, что вся
сетевая часть работает.

</details>


Коды пятисотой группы сообщают о поломке на той стороне: `500` — упал сам
обработчик, `502` и `504` приходят от обратного прокси перед приложением,
который либо получил от него мусор (`502`), либо не дождался ответа (`504`).
Для диагностики важно, что все они — уже ответ приложения: сеть отработала.

Ключ `-w` умеет показывать не только код, но и тайминги по этапам. Это
позволяет ответить на вопрос «а где, собственно, тормозит»:


In [ ]:
%%bash
curl -s -o /dev/null -m 10 https://example.com/ \
  -w 'имя в адрес: %{time_namelookup}\nсоединение:  %{time_connect}\nTLS:         %{time_appconnect}\nпервый байт: %{time_starttransfer}\nвсего:       %{time_total}\n'


Читается по разностям, а не по абсолютным числам: большой `time_namelookup` —
тормозит DNS; `time_appconnect - time_connect` — цена TLS-рукопожатия;
`time_starttransfer - time_appconnect` — столько думало само приложение. Для
`http://` без TLS `time_appconnect` равен нулю, и приложению достаётся весь
промежуток от `time_connect`.

А `-v` показывает сам разговор: куда пошли, какой адрес выбрали, что отправили
и что получили в ответ.


In [ ]:
%%bash
curl -v -s -o /dev/null -m 5 http://127.0.0.1:8000/ 2>&1 | head -12


В выводе `curl -v` каждая строка относится к своей ступени лестницы, и по
префиксу видно, кто говорит:

- `* Trying 127.0.0.1:8000...` — ступень 3–4, пробуем достучаться;
- `* Connected to ...` — ступень 4, TCP-соединение установлено;
- `> GET / HTTP/1.1` — строки со стрелкой вправо мы **отправили**, это ступень 5;
- `< HTTP/1.0 200 OK` — стрелка влево, ответ приложения (наш сервер отвечает
  версией 1.0 — это видно в первой строке ответа).

Строк про разрешение имени здесь нет: адрес задан числом, разрешать было нечего.
Не будет и блока TLS — схема `http://`, а не `https://`.


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`curl` и `wget` часто путают. Грубое разделение: `curl` — «разговор с
сервисом», по умолчанию печатает ответ в stdout, умеет любые методы и
заголовки, удобен в конвейере; `wget` — «скачать файл», умеет рекурсивный
обход и докачку, по умолчанию пишет на диск. Для диагностики берут `curl`,
для «утащить архив с датасетом» — `wget`.

Работу с REST API и разбор JSON мы будем разбирать отдельно в семинаре 11;
сегодня `curl` нужен нам только как измерительный прибор.

</details>


Напоследок — то, на чём держатся задачи M7 и H6. HTTP это просто текст,
который можно отправить чем угодно, хоть тем же `nc`. Формат запроса: строка
`МЕТОД ПУТЬ ВЕРСИЯ`, заголовки по строке на каждый и **пустая строка** в конце
— именно она говорит серверу, что заголовки кончились.


In [ ]:
%%bash
printf 'GET / HTTP/1.0\r\n\r\n' | nc -w 3 127.0.0.1 8000 | head -5


`\r\n` — перевод строки, который требует протокол (возврат каретки плюс
перевод строки), поэтому берём `printf`, а не `echo`. В ответе — та же строка
статуса и заголовки, что показывал `curl -I`: для сервера нет разницы, кто на
том конце.


## 11. Что видно в трафике: HTTP открытым текстом

До сих пор нас волновало, дошёл ли запрос. Теперь — **кто ещё прочитал его по
дороге**. HTTP передаёт всё как есть: путь, заголовки, куки, тело, пароль из
формы входа, токен в `Authorization`. Это видит каждый, через чьё оборудование
проходит пакет: точка доступа в кафе, провайдер, администратор сети, сосед по
коммутатору.

Убедимся своими глазами. `tcpdump` показывает пакеты, проходящие через
выбранный интерфейс; команда требует `sudo`, поэтому её выполняет
преподаватель.

Ключи: `-i lo` — слушать интерфейс `lo`, тот самый, через который ходит наш
учебный сервер; `-A` — печатать содержимое пакетов текстом, а не в
шестнадцатеричном виде; `-n` — не превращать адреса и порты в имена; `-q` —
короткие строки заголовков, чтобы текст не тонул в служебных полях. В конце —
фильтр `tcp port 8000`: берём только интересующее нас соединение.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
sudo -n true 2>/dev/null || { echo "нужен sudo — команду показывает преподаватель"; exit 0; }
sudo timeout 5 tcpdump -i lo -A -n -q 'tcp port 8000' > dump-http.txt 2>/dev/null &
sleep 1
curl -s -H 'Authorization: Bearer SUPERSECRET123' http://127.0.0.1:8000/ > /dev/null
sleep 5
grep -a -E 'GET /|Authorization' dump-http.txt


Токен лежит в дампе ровно так, как мы его отправили. Никакой расшифровки не
потребовалось — расшифровывать было нечего.

#### ❓ **Вопрос**: Мы слушали собственный интерфейс своей же машины. Кому этот токен был бы виден, если бы запрос ушёл в интернет?

<details>

<summary><strong>Ответ</strong></summary>

Всем, через кого идёт трафик: точке доступа Wi-Fi, её владельцу, провайдеру,
любому оборудованию по пути. Никакого взлома для этого не нужно — достаточно
той же команды `tcpdump`, запущенной на своей стороне канала.

Практический вывод: по HTTP нельзя отправлять ничего, что не готовы написать на
открытке. Пароли, токены, персональные данные, содержимое личного кабинета — всё
это требует HTTPS.

</details>


## 12. HTTPS: сертификат, удостоверяющий центр, MITM

HTTPS — это тот же HTTP, но внутри TLS-соединения. TLS решает сразу две задачи,
и путать их не стоит:

1. **шифрование** — по дороге содержимое никто не прочитает;
2. **аутентификация сервера** — мы разговариваем именно с `example.com`, а не с
   тем, кто перехватил соединение.

Без второй первая бессмысленна: шифровать канал до злоумышленника, который
представился нужным сайтом, незачем. Именно такая атака называется
**MITM** (man-in-the-middle, «человек посередине»).

Аутентификацию обеспечивает **сертификат** — файл, где написано «этот открытый
ключ принадлежит `example.com`» и стоит подпись **удостоверяющего центра**
(УЦ, англ. CA). Посмотрим на настоящий — ключ `-servername` передаёт серверу
имя сайта, к которому мы обращаемся (поле называется SNI, вернёмся к нему в
конце раздела).


In [ ]:
%%bash
timeout 15 openssl s_client -connect example.com:443 -servername example.com \
  < /dev/null 2>/dev/null | grep -E '^ *[0-9]+ s:| *i:|Verify return code'


Читается лесенкой: `s:` — кому выдан сертификат, `i:` — кто его выдал. Каждый
следующий подписан предыдущим, и цепочка упирается в **корневой** сертификат
УЦ. Строка `Verify return code: 0 (ok)` означает, что цепочка проверена
успешно.

Корень не приходит из сети — он уже лежит у нас в системе, в наборе доверенных
УЦ, который обновляется вместе с пакетами.


In [ ]:
%%bash
timeout 15 openssl s_client -connect example.com:443 -servername example.com \
  < /dev/null 2>/dev/null | openssl x509 -noout -subject -issuer -dates
echo "корневых УЦ в системе: $(grep -c 'BEGIN CERTIFICATE' /etc/ssl/certs/ca-certificates.crt)"


#### ❓ **Вопрос**: Мы первый раз в жизни зашли на сайт. Почему браузер сразу ему верит, никого не спрашивая?

<details>

<summary><strong>Ответ</strong></summary>

Он верит не сайту, а цепочке подписей. Сертификат сайта подписан
промежуточным УЦ, тот — корневым, а корневой уже лежит в системном хранилище,
которое приехало вместе с операционной системой или браузером. Доверие
переносится по цепочке: доверяю корню → доверяю тому, кого он подписал.

Проверяется при этом три вещи сразу: подпись цепочки, **имя** (сертификат
выписан именно на то имя, к которому мы обратились) и **срок действия**. Хватит
провалиться любой из трёх — и соединение будет разорвано.

</details>


Теперь встанем на место «человека посередине». У него нет подписи настоящего
УЦ — предъявить он может только сертификат, который выписал сам себе.
Изготовим такой и поднимем на нём HTTPS.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
openssl req -x509 -newkey rsa:2048 -nodes -days 365 \
  -keyout key.pem -out cert.pem -subj "/CN=localhost" \
  -addext "subjectAltName=DNS:localhost" 2>/dev/null
openssl x509 -in cert.pem -noout -subject -issuer


Флаги по одному: `-x509` — сразу готовый сертификат, а не запрос на подпись в
УЦ; `-newkey rsa:2048` — заодно сгенерировать новый ключ; `-nodes` — не
шифровать его паролем, иначе сервер спросит пароль при каждом запуске;
`-days 365` — срок действия; `-subj "/CN=localhost"` — на какое имя;
`-addext "subjectAltName=DNS:localhost"` — то же имя в поле **SAN**.

Поле SAN здесь не формальность: современные клиенты проверяют имя именно по
нему, а на `CN` уже не смотрят. Сертификат без SAN будет отвергнут с ошибкой о
несовпадении имени, даже если `CN` правильный. Чтобы выписать сертификат на
другое имя, меняют оба места.

`subject` и `issuer` в выводе совпали: сертификат сам себе удостоверяющий
центр. Это и называется самоподписанным.

Сервер поднимем встроенным режимом `s_server`. Ключ `-www` заставляет его
отвечать простой страницей и **продолжать принимать соединения** — без него
`s_server` обслужил бы одно и вышел, а нам нужно несколько запросов подряд. Поднимем на нём сервер — у `openssl` для этого есть
встроенный режим `s_server`.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
pkill -f 's_server -accept 8443'
openssl s_server -accept 8443 -cert cert.pem -key key.pem -www \
  > s_server.log 2>&1 &
sleep 1
ss -tln | grep 8443


In [ ]:
%%bash
curl -sS -m 5 -o /dev/null https://localhost:8443/
echo "код возврата curl: $?"


`curl` отказался разговаривать: код `60`, «SSL certificate problem». Обратите
внимание — сервер жив, порт слушается, шифрование работает. Отказ произошёл на
проверке подлинности, и это не поломка, а сработавшая защита: ровно так
выглядит попытка MITM.

Способов договориться два, и они принципиально разные.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
curl -sk -m 5 -o /dev/null -w 'с -k (не проверять):    %{http_code}\n' https://localhost:8443/
curl -s -m 5 --cacert cert.pem -o /dev/null \
  -w 'с --cacert (доверяем): %{http_code}\n' https://localhost:8443/


#### ❓ **Вопрос**: Оба вызова вернули `200`. Почему `-k` в рабочем скрипте — плохая идея, а `--cacert` — нормальная?

<details>

<summary><strong>Ответ</strong></summary>

`-k` (`--insecure`) отключает проверку целиком: клиент примет **любой**
сертификат от кого угодно. Шифрование останется, аутентификация исчезнет, и
атака «человек посередине» снова становится возможной — а скрипт при этом
работает как ни в чём не бывало, поэтому проблему никто не заметит.

`--cacert` не отключает проверку, а расширяет доверие ровно на один известный
нам сертификат. Все остальные по-прежнему будут отвергнуты. Так и надо
подключаться к внутренним сервисам с собственным УЦ.

</details>


Осталось проверить главное обещание TLS. Повторим перехват трафика — тот же
токен, тот же `tcpdump`, только порт теперь HTTPS-овый.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
sudo -n true 2>/dev/null || { echo "нужен sudo — команду показывает преподаватель"; exit 0; }
sudo timeout 5 tcpdump -i lo -A -n -q 'tcp port 8443' > dump-tls.txt 2>/dev/null &
sleep 1
curl -sk -m 5 -H 'Authorization: Bearer SUPERSECRET123' https://localhost:8443/ > /dev/null
sleep 5
echo "токен в дампе:     $(grep -ac SUPERSECRET dump-tls.txt) раз"
echo "имя сервера (SNI): $(grep -ac localhost dump-tls.txt) раз"


Токена нет ни разу — в дампе один шифрованный мусор. А вот имя сервера видно:
клиент сообщает его открытым текстом в начале рукопожатия (расширение **SNI**),
иначе сервер с несколькими сайтами на одном адресе не поймёт, чей сертификат
предъявлять.

Сколько именно раз встретится имя, зависит от версии протокола. В TLS 1.3 —
один: сертификат сервера уже шифруется. В TLS 1.2 сертификат идёт открытым
текстом, и имя попадётся ещё и в нём. Важно не число, а факт: имя сервера
скрыть не удаётся ни в одной версии.

Что TLS **не** скрывает: факт соединения, адреса сторон, объём и время передачи,
имя сервера. Что скрывает: путь, заголовки, куки, тело — всё содержимое запроса
и ответа.

#### ❓ **Вопрос**: Сервис лежит на публичном адресе и работает по HTTP. Что плохого случится, если это «просто внутренняя админка для своих»?

<details>

<summary><strong>Ответ</strong></summary>

Как минимум три вещи. Пароль или токен уйдёт по сети открытым текстом — мы это
только что видели в дампе. Клиент не может проверить, что отвечает именно ваш
сервер, — подменить ответ по дороге ничто не мешает. И содержимое можно не
только прочитать, но и **изменить**: HTTP не защищён от подмены на лету.

«Внутренняя» и «публичная» — разные вещи: если у сервиса есть публичный адрес,
он публичный, и его найдут сканеры. Правильный минимум сегодня: сертификат от
Let's Encrypt (бесплатно, выпускается и продлевается автоматически), редирект с
HTTP на HTTPS и закрытый фаерволом всё остальное.

</details>


Учебный HTTPS-сервер больше не нужен — снимаем его.


In [ ]:
%%bash
pkill -f 's_server -accept 8443'
sleep 1
ss -tln | grep 8443 || echo "порт 8443 свободен"


## 13. Лестница целиком

Итоговая таблица — с ней и надо подходить к недоступному сервису:

| Что видим | Ступень | Чем проверяем | Куда смотреть дальше |
|---|---|---|---|
| `Name or service not known` | 2. имя | `getent hosts`, `dig` | опечатка в имени, `/etc/hosts`, `/etc/resolv.conf` |
| `dig` отвечает, программа идёт не туда | 2. имя | `getent hosts` | перекрывающая запись в `/etc/hosts` |
| `ping` молчит, но сервис работает | 3. хост | `nc -z -v -w 3 ХОСТ ПОРТ` | нормально: ICMP режут, проверяем сразу порт |
| `Connection refused` | 4. порт | `ss -tlnp` на сервере, затем `ufw status` | сервис не запущен, слушает другой порт — либо отказ прислал фаервол с REJECT |
| Локально работает, снаружи `refused` | 4. порт | `ss -tlnp`, колонка адреса | привязка к `127.0.0.1` вместо `0.0.0.0` |
| Таймаут, соединение так и не встало | 3–4. хост или порт | `ping`, затем `ufw status` и облачные правила | по одному таймауту эти две ступени не различить: молчат одинаково |
| Соединение мгновенно, ответа нет | 5. приложение | `curl -w '%{time_connect}'`, логи | сеть в порядке, думает само приложение |
| `Empty reply from server` (код 52) | 5. приложение | логи сервиса | сервер оборвал соединение сам |
| `504 Gateway Timeout` | 5. приложение | логи прокси и приложения | прокси не дождался ответа от приложения |
| `SSL certificate problem` | 5. приложение | `openssl s_client`, `curl -v` | сертификат самоподписанный, просрочен или не на то имя |
| `404`, `500`, `502` | 5. приложение | `curl -I`, логи сервиса | любой осмысленный код — это уже ответ приложения: сеть в порядке |
| Всё открывается, но медленно | 5. приложение | `curl -w` тайминги | смотрим, какой этап съел время |

Правило одно: **первая ступень, которая ответила не так, как ожидалось, и есть
место поломки**. Подниматься выше неё бессмысленно.


### Убираем за собой

Учебный сервер нам больше не нужен — снимаем его, чтобы порт не остался занят.


In [ ]:
%%bash
pkill -f "http.server 8000"
sleep 1
ss -tlnp | grep 8000 || echo "порт 8000 свободен"


## Что осталось за кадром

- **SSH: ключи, `~/.ssh/config`, туннели** — семинар 8. Там же приём
  «пробросить порт с сервера к себе», когда сервис намеренно слушает только
  loopback.
- **Проброс портов контейнера, сети docker** — семинар 9.
- **REST API, разбор JSON, `requests`** — семинар 11. Сегодня `curl` был
  измерительным прибором, а не клиентом API.
- **`traceroute` и `mtr`** — когда таймаут случается не у вас и не на сервере,
  а где-то посередине маршрута.
- **Wireshark и разбор протоколов по байтам** — `tcpdump` мы сегодня трогали
  только чтобы увидеть разницу между HTTP и HTTPS. Полноценный анализ трафика —
  следующий уровень, но начинается он всё равно с той же лестницы.
- **Свой удостоверяющий центр и Let's Encrypt** — сегодня мы выписали
  самоподписанный сертификат, чтобы увидеть отказ клиента. Как получить
  настоящий и продлевать его автоматически — отдельная тема.
